# 07 — Exploratory analysis: the two leaks

**The Democratic Funnel** · Team UL · DIRISA Student Datathon Challenge 2026

**Owner:** Role F, with everyone  **Reads:** `data/processed/master.csv`  **Writes:** `reports/figures/`

---

## What we are looking for

Every municipality loses people at two points on the way to the ballot box:

```
Adults who may vote  →  People who registered  →  People who voted
        E                        R                       V
            registration leak         turnout leak
```

- **r = R / E** — registration rate. How full is the roll?
- **t = V / R** — turnout. Of those on the roll, how many voted?
- **p = r × t** — real participation. Of all adults, how many voted?

The turnout figure in the news is only `t`. It counts nobody who never registered.

**Every chart below answers one question: which leak, where, and how big?**

Each section has the chart, then one line saying what it shows. Those lines become
the slides.

## 0. Setup

In [ ]:
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Thato20-M/democratic-funnel.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/democratic-funnel")
    url = REPO_URL
    try:
        from google.colab import userdata
        url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:
        print("No GITHUB_TOKEN in Colab Secrets - cloning will fail for the private repo")
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", url, str(REPO_DIR)], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "src" / "config.py").exists())

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Repo:", REPO_DIR)

In [ ]:
from src import config
from src.config import RAW, INTERIM, PROCESSED, FIGURES, SEED

import numpy as np, pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings; warnings.filterwarnings("ignore", category=FutureWarning)

config.make_dirs(); config.set_seed()
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 170)
print(config.describe())

### House style for every chart

One palette, used the same way everywhere, so the slides read as one deck.

| Colour | Means |
|---|---|
| **Blue** | registration problem |
| **Yellow** | turnout problem |
| **Red** | both problems |
| **Grey** | healthy, or background context |

The three hues were checked for colour-blind separation before use (all-pairs
ΔE 15.3 under deuteranopia, well clear of the ≥8 target). Yellow is light against a white
background, so **every chart that uses it also carries a written label** — colour never
carries meaning on its own.

In [ ]:
# --- palette -------------------------------------------------------------
BLUE, YELLOW, RED = "#2a78d6", "#eda100", "#e34948"
GREY, GRID        = "#b8b7b2", "#e6e5e1"
INK, INK2, MUTED  = "#0b0b0b", "#52514e", "#8a8f96"
SURFACE           = "#fcfcfb"

LEAK_COLOURS = {"Healthy": GREY, "Registration leak": BLUE,
                "Turnout leak": YELLOW, "Both leaks": RED}

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.linewidth": 0.8,
    "axes.labelcolor": INK2, "axes.titlecolor": INK,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.titlepad": 12, "axes.labelsize": 9,
    "xtick.color": INK2, "ytick.color": INK2,
    "xtick.labelsize": 8.5, "ytick.labelsize": 8.5,
    "grid.color": GRID, "grid.linewidth": 0.7,
    "legend.frameon": False, "legend.fontsize": 9,
    "font.size": 9.5,
})

def tidy(ax, xgrid=False, ygrid=True):
    """Recessive axes: no box, grid behind the data only where it helps reading."""
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(GRID)
    ax.set_axisbelow(True)
    ax.grid(ygrid, axis="y"); ax.grid(xgrid, axis="x")
    return ax

def pct(ax, axis="y"):
    (ax.yaxis if axis == "y" else ax.xaxis).set_major_formatter(
        mtick.FuncFormatter(lambda v, _: f"{v:.0%}"))
    return ax

def save(fig, name):
    fig.savefig(FIGURES / f"{name}.png", bbox_inches="tight")
    print("saved", name)

## 1. Load the data

`master.csv` has one row per municipality per election: 213 municipalities × 4 elections
(2011, 2016, 2021, 2026) = 852 rows.

2026 has registration figures but no votes — that election has not happened yet. So
anything involving turnout is blank for 2026 until the model fills it in.

In [ ]:
master = pd.read_csv(PROCESSED / "master.csv")
bounds = gpd.read_file(INTERIM / "boundaries.gpkg")

print(master.shape)
print(master.groupby("election").size().to_string())

# handy slices
y21 = master[master.election == 2021].copy()
y26 = master[master.election == 2026].copy()
hist = master[master.election < 2026].copy()      # elections actually held

---
## 2. The headline: the two leaks move in opposite directions

This is the single most important table in the project.

Read down the columns. Registration has been getting **better** every election. Turnout has
been getting **worse**. Real participation therefore went up, then down.

In [ ]:
trend = (hist.groupby("election")[["r", "t", "p"]].median()
             .reindex([2011, 2016, 2021]))
trend.loc[2026, "r"] = y26["r"].median()

print("Median municipality:")
print((trend * 100).round(1).to_string())

### The same thing as a chart

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))

yrs = [2011, 2016, 2021]
ax.plot(yrs, trend.loc[yrs, "r"], "-o", color=BLUE,   lw=2, ms=7, label="Registration rate  r = R/E")
ax.plot(yrs, trend.loc[yrs, "t"], "-o", color=YELLOW, lw=2, ms=7, label="Turnout  t = V/R")
ax.plot(yrs, trend.loc[yrs, "p"], "-o", color=GREY,   lw=2, ms=7, label="Real participation  p = r x t")

# 2026 registration is known; turnout is not
ax.plot([2021, 2026], trend.loc[[2021, 2026], "r"], "--o", color=BLUE, lw=2, ms=7, alpha=.9)
ax.annotate("2026 roll\n(record high)", (2026, trend.loc[2026, "r"]),
            xytext=(-6, 14), textcoords="offset points", ha="right",
            fontsize=8.5, color=BLUE, weight="bold")
ax.annotate("? not yet held", (2026, trend.loc[2021, "t"]), xytext=(4, -2),
            textcoords="offset points", fontsize=8.5, color=MUTED, style="italic")

for lbl, col, key in [("r", BLUE, "r"), ("t", YELLOW, "t"), ("p", GREY, "p")]:
    for yr in yrs:
        ax.annotate(f"{trend.loc[yr, key]:.0%}", (yr, trend.loc[yr, key]),
                    xytext=(0, 9), textcoords="offset points",
                    ha="center", fontsize=8, color=INK2)

ax.set_xticks([2011, 2016, 2021, 2026]); ax.set_xlim(2009, 2028)
ax.set_ylim(0.25, 0.9); pct(ax); tidy(ax)
ax.set_title("The roll keeps filling. Voting keeps falling.")
ax.set_ylabel("median municipality")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncols=3)
save(fig, "01_two_leaks_trend"); plt.show()

**What it shows.** Registration climbed from 63% to a record 79% of eligible adults,
while turnout fell from 60% to 48%. In 2021 the fullest roll yet produced the worst
participation of the three elections.

**Why it matters.** If you only watch turnout you would say local democracy is collapsing. If
you only watch registration you would say it is improving. Both are true. They are *different
problems*, and the standard turnout figure shows you only one of them.

*Note: the 2026 registration rate uses a 2022 population denominator, so it is overstated by
roughly 5%. The record high holds either way.*

---
## 3. The funnel itself

The same three numbers as a funnel: of every 100 adults who may vote, how many register, and
how many then vote?

In [ ]:
def funnel_counts(df):
    E = df["eligible_adults"].sum()
    R = df["registered"].sum()
    V = df["votes_cast"].sum() if df["votes_cast"].notna().any() else np.nan
    return E, R, V

E21, R21, V21 = funnel_counts(y21)
E26, R26, _   = funnel_counts(y26)

fig, ax = plt.subplots(figsize=(7.4, 3.4))
stages = ["May vote\n(adults)", "Registered", "Voted"]
vals21 = [100, 100 * R21 / E21, 100 * V21 / E21]
vals26 = [100, 100 * R26 / E26, np.nan]

ypos = np.arange(3)[::-1]
ax.barh(ypos + .18, vals21, height=.32, color=GREY, label="2021 (actual)")
ax.barh(ypos - .18, vals26, height=.32, color=BLUE, label="2026 (roll so far)")

for y, v in zip(ypos + .18, vals21):
    ax.annotate(f"{v:.0f}", (v, y), xytext=(5, 0), textcoords="offset points",
                va="center", fontsize=9, color=INK2)
for y, v in zip(ypos - .18, vals26):
    if not np.isnan(v):
        ax.annotate(f"{v:.0f}", (v, y), xytext=(5, 0), textcoords="offset points",
                    va="center", fontsize=9, color=BLUE, weight="bold")
ax.annotate("not yet held", (2, ypos[2] - .18), xytext=(4, 0),
            textcoords="offset points", va="center", fontsize=8.5,
            color=MUTED, style="italic")

ax.set_yticks(ypos); ax.set_yticklabels(stages)
ax.set_xlim(0, 112); ax.set_xlabel("per 100 adults who may vote")
tidy(ax, xgrid=True, ygrid=False)
ax.set_title("Out of every 100 adults, how many actually vote?")
ax.legend(loc="lower right")
save(fig, "02_funnel_stages"); plt.show()

print(f"2021: {100*R21/E21:.0f} of 100 adults registered, {100*V21/E21:.0f} voted")
print(f"2026: {100*R26/E26:.0f} of 100 adults registered so far")

**What it shows.** In 2021, about 65 of every 100 eligible adults were registered, and
only about 30 voted. Roughly 35 adults were lost before registration, and another 35 after it.

**Why it matters.** The two losses are about the same size. Half the problem is invisible in
the official turnout figure.

---
## 4. Turnout over time, by province

National average in bold; each province as a thin line behind it.

In [ ]:
prov = (hist.groupby(["election", "province"])
            .apply(lambda d: d.votes_cast.sum() / d.registered.sum())
            .unstack())
nat = hist.groupby("election").apply(lambda d: d.votes_cast.sum() / d.registered.sum())

fig, ax = plt.subplots(figsize=(7, 4.2))
for p in prov.columns:
    ax.plot(prov.index, prov[p], color=GREY, lw=1.2, alpha=.85)
    ax.annotate(p, (prov.index[-1], prov[p].iloc[-1]), xytext=(4, 0),
                textcoords="offset points", va="center", fontsize=7.5, color=MUTED)
ax.plot(nat.index, nat.values, "-o", color=BLUE, lw=2.5, ms=7, label="National", zorder=5)
for x, v in nat.items():
    ax.annotate(f"{v:.0%}", (x, v), xytext=(0, -16), textcoords="offset points",
                ha="center", fontsize=8.5, color=BLUE, weight="bold")

ax.set_xticks([2011, 2016, 2021]); ax.set_xlim(2010, 2024)
pct(ax); tidy(ax)
ax.set_title("Turnout fell in every province in 2021")
ax.set_ylabel("votes cast / registered"); ax.legend(loc="lower left")
save(fig, "03_turnout_by_province"); plt.show()

print((prov * 100).round(1).to_string())

**What it shows.** National turnout was flat between 2011 and 2016, then fell about 12
points in 2021. Every province fell.

**Why it matters.** The 2021 drop was nationwide, not local. That points at a national cause —
the election was held under COVID conditions — and it is the reason we predict the *change* in
turnout rather than the level.

---
## 5. How much do municipalities differ?

National averages hide the spread. These are all 213 municipalities.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))

ax = axes[0]
ax.hist(y21["r"].dropna(), bins=28, color=GREY, label="2021")
ax.hist(y26["r"].dropna(), bins=28, color=BLUE, alpha=.75, label="2026")
ax.axvline(1.0, color=RED, lw=1.2, ls="--")
ax.annotate("roll larger than\nthe adult population", (1.0, ax.get_ylim()[1]*.75),
            xytext=(6, 0), textcoords="offset points", fontsize=8, color=RED)
pct(ax, "x"); tidy(ax)
ax.set_title("Registration rate  r"); ax.set_xlabel("registered / eligible adults")
ax.set_ylabel("municipalities"); ax.legend()

ax = axes[1]
for yr, col, lab in [(2011, GREY, "2011"), (2016, "#9ec5f4", "2016"), (2021, BLUE, "2021")]:
    ax.hist(master.loc[master.election == yr, "t"].dropna(), bins=28,
            histtype="step", lw=2, color=col, label=lab)
pct(ax, "x"); tidy(ax)
ax.set_title("Turnout  t"); ax.set_xlabel("votes cast / registered")
ax.set_ylabel("municipalities"); ax.legend()

fig.suptitle("The spread across municipalities is wide", x=0.09, ha="left",
             fontsize=12, weight="bold", y=1.04)
save(fig, "04_distributions"); plt.show()

print(master.groupby("election")[["r", "t"]].describe().round(3).T.to_string())

**What it shows.** Turnout ranges from roughly 30% to 70% between municipalities, and the
whole 2021 distribution shifted left. A handful of municipalities have `r` above 1 — more
registered voters than the census counted adults.

**Why it matters.** A national figure is useless for targeting. The point of this project is
the spread, not the average.

**On `r` above 1:** the voters' roll is *cumulative* — people stay on it after they move away
or die. The census is a *point-in-time* count. In municipalities losing population, the roll
drifts above the resident adult population. We flag these rather than deleting them; they are
mostly Northern Cape and mining towns.

---
## 6. The key chart: which leak does each municipality have?

For each municipality we compare it to the national median of that election:

- **registration leak** = ln(r / median r) — negative means fewer people registered than typical
- **turnout leak** = ln(t / median t) — negative means fewer registered people voted than typical

Taking logs makes the two add up exactly: a municipality's total participation shortfall is its
registration leak **plus** its turnout leak. No model, no assumptions — just arithmetic.

That splits every municipality into one of four groups.

In [ ]:
def classify(d):
    out = pd.Series("Healthy", index=d.index)
    reg_bad, tur_bad = d["leak_reg"] < 0, d["leak_turn"] < 0
    out[reg_bad & ~tur_bad] = "Registration leak"
    out[~reg_bad & tur_bad] = "Turnout leak"
    out[reg_bad & tur_bad]  = "Both leaks"
    return out

y21["leak_type"] = classify(y21)
print(y21["leak_type"].value_counts().to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(6.6, 6.2))

for name, col in LEAK_COLOURS.items():
    s = y21[y21.leak_type == name]
    ax.scatter(s["leak_reg"], s["leak_turn"], s=34, color=col,
               edgecolor=SURFACE, linewidth=.7, label=f"{name}  ({len(s)})")

ax.axhline(0, color=INK2, lw=1); ax.axvline(0, color=INK2, lw=1)
lim = 0.62
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)

# quadrant corners: x = registration leak, y = turnout leak
quad = [(-lim*.94,  lim*.86, "REGISTRATION LEAK", "not registered - but those\nwho were, did vote", "left"),
        ( lim*.94,  lim*.86, "HEALTHY",           "above the median\non both",                     "right"),
        (-lim*.94, -lim*.86, "BOTH LEAKS",        "few registered, and few\nof those voted",       "left"),
        ( lim*.94, -lim*.86, "TURNOUT LEAK",      "the roll is full - but\nthey did not vote",     "right")]
for x, y, head, sub, ha in quad:
    ax.text(x, y, head, fontsize=9, color=INK, weight="bold", ha=ha, va="bottom")
    ax.text(x, y, sub, fontsize=8, color=MUTED, ha=ha, va="top", linespacing=1.4)

ax.set_xlabel("registration leak        worse \u2190        \u2192 better")
ax.set_ylabel("turnout leak        worse \u2193        \u2191 better")
tidy(ax, xgrid=True)
ax.set_title("2021: every municipality has one of four problems")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.11), ncols=2)
save(fig, "05_leak_quadrants_2021"); plt.show()

**What it shows.** Municipalities scatter into four clear groups. Some have a full roll and
poor turnout. Others have poor registration but loyal voters. Some have both.

**Why it matters — this is the whole argument.** Two municipalities can report the same turnout
and sit in different quadrants. They need **opposite** responses:

| Quadrant | What to send |
|---|---|
| Registration leak | registration drives, ID and address help |
| Turnout leak | voter education, mobilisation, election-day access |
| Both leaks | first priority for everything |
| Healthy | nothing urgent |

Nobody can make that call from the published turnout figure alone.

---
## 7. Where to act before 4 November 2026

2026 has no turnout yet, so we combine **what we know now** (registration for 2026) with
**what happened last time** (turnout in 2021).

The top-left corner is the priority list: the roll is thin *and* the people on it did not turn
out last time.

In [ ]:
prio = y26[["muni_code", "muni_name", "province", "leak_reg",
            "registered", "eligible_adults", "registered_youth", "eligible_youth"]].copy()
prio = prio.merge(y21[["muni_code", "leak_turn", "t"]], on="muni_code", how="left")
prio["priority"] = -(prio["leak_reg"] + prio["leak_turn"])       # bigger = worse

fig, ax = plt.subplots(figsize=(6.8, 5.6))
worst = prio.nlargest(15, "priority")

ax.scatter(prio["leak_reg"], prio["leak_turn"], s=30, color=GREY,
           edgecolor=SURFACE, linewidth=.7, label="all municipalities")
ax.scatter(worst["leak_reg"], worst["leak_turn"], s=52, color=RED,
           edgecolor=SURFACE, linewidth=.8, label="15 highest priority", zorder=4)
for _, row in worst.head(8).iterrows():
    ax.annotate(row["muni_name"][:22], (row["leak_reg"], row["leak_turn"]),
                xytext=(5, 4), textcoords="offset points", fontsize=7.5, color=INK2)

ax.axhline(0, color=INK2, lw=1); ax.axvline(0, color=INK2, lw=1)
ax.set_xlabel("registration leak, 2026 roll  \u2190 worse")
ax.set_ylabel("turnout leak, 2021  \u2190 worse")
tidy(ax, xgrid=True)
ax.set_title("Priority for the weeks before 4 November")
ax.legend(loc="upper left")
save(fig, "06_priority_2026"); plt.show()

print(worst[["muni_code", "muni_name", "province", "leak_reg", "leak_turn"]]
      .round(3).to_string(index=False))

**What it shows.** Fifteen municipalities are below the national median on both measures.

**Why it matters.** This is the deliverable an IEC or civil-society planner can act on. It is a
list of places, not a national statistic.

*This ranking is based on 2021 turnout. The model in `08_model` replaces it with predicted 2026
turnout.*

---
## 8. The worst 20 on each leak

Same information, ranked, for the report appendix.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 6))

w_reg = y26.nsmallest(20, "leak_reg").sort_values("leak_reg")
ax = axes[0]
ax.barh(range(20), w_reg["r"], color=BLUE, height=.72)
ax.set_yticks(range(20))
ax.set_yticklabels([f"{n[:26]}" for n in w_reg["muni_name"]])
ax.axvline(y26["r"].median(), color=INK2, lw=1.2, ls="--")
ax.annotate(f"national median {y26['r'].median():.0%}",
            (y26["r"].median(), 19.6), xytext=(4, 0), textcoords="offset points",
            fontsize=8, color=INK2)
for i, v in enumerate(w_reg["r"]):
    ax.annotate(f"{v:.0%}", (v, i), xytext=(4, 0), textcoords="offset points",
                va="center", fontsize=7.5, color=INK2)
pct(ax, "x"); tidy(ax, xgrid=True, ygrid=False)
ax.set_title("Thinnest rolls, 2026"); ax.set_xlabel("registered / eligible adults")

w_tur = y21.nsmallest(20, "leak_turn").sort_values("leak_turn")
ax = axes[1]
ax.barh(range(20), w_tur["t"], color=YELLOW, height=.72)
ax.set_yticks(range(20))
ax.set_yticklabels([f"{n[:26]}" for n in w_tur["muni_name"]])
ax.axvline(y21["t"].median(), color=INK2, lw=1.2, ls="--")
ax.annotate(f"national median {y21['t'].median():.0%}",
            (y21["t"].median(), 19.6), xytext=(4, 0), textcoords="offset points",
            fontsize=8, color=INK2)
for i, v in enumerate(w_tur["t"]):
    ax.annotate(f"{v:.0%}", (v, i), xytext=(4, 0), textcoords="offset points",
                va="center", fontsize=7.5, color=INK2)
pct(ax, "x"); tidy(ax, xgrid=True, ygrid=False)
ax.set_title("Lowest turnout, 2021"); ax.set_xlabel("votes cast / registered")

fig.tight_layout()
save(fig, "07_worst20"); plt.show()

**What it shows.** Two different lists of municipalities. Very little overlap.

**Why it matters.** It proves the two leaks are not the same thing measured twice.

---
## 9. The youth registration gap

Young people are the clearest case of a registration leak, and the IEC publishes registration
by age for 2026 — so we can measure it directly.

We compare **registered 18–29** against **eligible 18–29** from Census 2022.

In [ ]:
y26["youth_r"]     = y26["registered_youth"] / y26["eligible_youth"]
y26["adult_r"]     = y26["registered"] / y26["eligible_adults"]
y26["youth_gap"]   = y26["eligible_youth"] - y26["registered_youth"]

EY, RY = y26["eligible_youth"].sum(), y26["registered_youth"].sum()
EA, R  = y26["eligible_adults"].sum(), y26["registered"].sum()

print(f"eligible 18-29            {EY:>14,.0f}   ({EY/EA:.0%} of adults)")
print(f"registered 18-29          {RY:>14,.0f}")
print(f"NOT registered, 18-29     {EY-RY:>14,.0f}")
print()
print(f"registration rate, 18-29  {RY/EY:>14.1%}")
print(f"registration rate, 30+    {(R-RY)/(EA-EY):>14.1%}")
print()
print(f"all unregistered adults   {EA-R:>14,.0f}")
print(f"  of whom aged 18-29      {(EY-RY)/(EA-R):>14.1%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6),
                         gridspec_kw={"width_ratios": [1.15, 1]})

# --- left: two stacked bars - who the adults are, and who is missing
ax = axes[0]
youth_of_adults  = EY / EA
youth_of_missing = (EY - RY) / (EA - R)

bars = [("All eligible adults", youth_of_adults, EY, EA - EY),
        ("Missing from the roll", youth_of_missing, EY - RY, (EA - R) - (EY - RY))]
for i, (lab, share, n_y, n_o) in enumerate(bars):
    y = 1 - i
    ax.barh(y, share, height=.46, color=BLUE)
    # 2px surface gap between the two segments
    ax.barh(y, 1 - share, left=share + 0.006, height=.46, color=GREY)
    ax.annotate(f"18-29\n{share:.0%}  ({n_y/1e6:.1f}m)", (share / 2, y),
                ha="center", va="center", fontsize=9, color="white", weight="bold")
    ax.annotate(f"30+\n{1-share:.0%}  ({n_o/1e6:.1f}m)", (share + (1 - share) / 2, y),
                ha="center", va="center", fontsize=9, color=INK2)

ax.set_yticks([1, 0]); ax.set_yticklabels([b[0] for b in bars], fontsize=9.5)
ax.set_xlim(0, 1.02); ax.set_xticks([]); ax.set_ylim(-0.6, 1.6)
for sp in ax.spines.values(): sp.set_visible(False)
ax.grid(False)
ax.set_title(f"{youth_of_adults:.0%} of adults, "
             f"{youth_of_missing:.0%} of everyone missing from the roll")

# --- right: youth vs overall rate per municipality
ax = axes[1]
ax.scatter(y26["adult_r"], y26["youth_r"], s=30, color=BLUE,
           edgecolor=SURFACE, linewidth=.7, alpha=.85)
lo, hi = 0.35, 1.35
ax.plot([lo, hi], [lo, hi], color=INK2, lw=1.2, ls="--")
ax.annotate("youth register as often\nas everyone else", (hi*.82, hi*.82),
            xytext=(-6, 10), textcoords="offset points", ha="right",
            fontsize=8, color=INK2)
ax.set_xlim(lo, hi); ax.set_ylim(0.1, hi)
pct(ax); pct(ax, "x"); tidy(ax, xgrid=True)
ax.set_title("Every municipality sits below the line")
ax.set_xlabel("registration rate, all adults"); ax.set_ylabel("registration rate, 18-29")

fig.tight_layout()
save(fig, "08_youth_gap"); plt.show()

**What it shows.** People aged 18–29 are about **30% of eligible adults but roughly 60% of
everyone missing from the voters' roll** — around 6.5 million young South Africans. Every single
municipality sits below the parity line.

**Why it matters.** This is the largest single component of the registration leak, it is the
same story everywhere in the country, and registration is something that can still be changed
before election day.

**What we cannot say.** The IEC does not publish *turnout* by age at municipal level. So this is
a finding about **registration**, not about whether young people vote. We do not claim the
second.

In [ ]:
worst_youth = y26.nsmallest(15, "youth_r")[
    ["muni_code", "muni_name", "province", "youth_r", "adult_r", "youth_gap"]]
print("Lowest youth registration rates:")
print(worst_youth.assign(youth_r=lambda d: (d.youth_r*100).round(1),
                         adult_r=lambda d: (d.adult_r*100).round(1),
                         youth_gap=lambda d: d.youth_gap.round(0))
      .to_string(index=False))

---
## 10. The map

The same four groups, on the country.

In [ ]:
gmap = bounds.merge(y21[["muni_code", "leak_type", "leak_reg", "leak_turn", "t", "r"]],
                    on="muni_code", how="left")

fig, ax = plt.subplots(figsize=(8, 8))
for name, col in LEAK_COLOURS.items():
    sub = gmap[gmap.leak_type == name]
    if len(sub):
        sub.plot(ax=ax, color=col, edgecolor=SURFACE, linewidth=.35,
                 label=f"{name}  ({len(sub)})")
ax.set_axis_off()
ax.set_title("2021: which leak, where", loc="left", fontsize=13, weight="bold")
ax.legend(loc="lower left", fontsize=9.5)
save(fig, "09_leak_map"); plt.show()

**What it shows.** The leak types cluster geographically rather than scattering at random.

**Why it matters.** Clustering means the problem has regional causes, and it justifies giving the
model a **spatial** feature — each municipality's neighbours' turnout.

---
## 11. Does low participation cluster?

We test the assumption behind the spatial feature directly: is a municipality's turnout related
to its neighbours' turnout?

In [ ]:
sp = y21.dropna(subset=["spatial_lag_turnout", "t"])
rho = sp["t"].corr(sp["spatial_lag_turnout"])

fig, ax = plt.subplots(figsize=(5.6, 5.2))
ax.scatter(sp["spatial_lag_turnout"], sp["t"], s=30, color=BLUE,
           edgecolor=SURFACE, linewidth=.7, alpha=.85)
b, a = np.polyfit(sp["spatial_lag_turnout"], sp["t"], 1)
xs = np.linspace(sp["spatial_lag_turnout"].min(), sp["spatial_lag_turnout"].max(), 50)
ax.plot(xs, a + b * xs, color=RED, lw=2)
ax.annotate(f"correlation = {rho:.2f}", (0.05, 0.93), xycoords="axes fraction",
            fontsize=10, color=RED, weight="bold")
pct(ax); pct(ax, "x"); tidy(ax, xgrid=True)
ax.set_xlabel("average turnout of neighbouring municipalities")
ax.set_ylabel("own turnout")
ax.set_title("Turnout clusters geographically")
save(fig, "10_spatial_clustering"); plt.show()

**What it shows.** A municipality's turnout is strongly related to its neighbours' turnout.

**Why it matters.** It confirms `spatial_lag_turnout` carries real information, and it is why we
built the neighbour list from the boundary geometry.

---
## 12. Metros against local municipalities

Eight metros hold a large share of the country's voters. Do they behave differently?

In [ ]:
lbl = {"A": "Metro (8)", "B": "Local (205)"}
fig, axes = plt.subplots(1, 3, figsize=(11, 3.8))

for ax, (col, title, yr) in zip(axes, [("r", "Registration rate, 2026", 2026),
                                       ("t", "Turnout, 2021", 2021),
                                       ("youth_r", "Youth registration, 2026", 2026)]):
    src = y26 if yr == 2026 else y21
    if col == "youth_r" and col not in src:
        continue
    data = [src.loc[src.muni_category == c, col].dropna() for c in ["A", "B"]]
    bp = ax.boxplot(data, patch_artist=True, widths=.5,
                    medianprops=dict(color=INK, lw=2),
                    flierprops=dict(marker="o", ms=4, mfc=GREY, mec=SURFACE))
    for patch, c in zip(bp["boxes"], [BLUE, GREY]):
        patch.set_facecolor(c); patch.set_edgecolor(SURFACE)
    ax.set_xticklabels([lbl["A"], lbl["B"]])
    pct(ax); tidy(ax)
    ax.set_title(title)

fig.suptitle("Metros register and vote differently", x=0.075, ha="left",
             fontsize=12, weight="bold", y=1.04)
save(fig, "11_metro_vs_local"); plt.show()

print(y26.groupby("muni_category")[["r", "youth_r"]].median().round(3).to_string())
print(y21.groupby("muni_category")[["t"]].median().round(3).to_string())

**What it shows.** Fill this in from your own output — compare the medians printed below the
chart and say in one sentence which way each measure goes.

**Why it matters.** If the youth gap is worse in metros, the recommendation is urban youth
campaigns, which is a different plan from rural registration drives.

---
## 13. What goes with low participation?

A first look at which municipal characteristics move together with the two leaks.

**This shows association, not cause.** A pattern across municipalities cannot prove how
individuals behave — that is the ecological fallacy, and we state it wherever these numbers
appear.

In [ ]:
feats = ["leak_reg", "leak_turn", "r", "t",
         "higher_ed_rate", "piped_water_rate", "electricity_rate", "refuse_removal_rate",
         "pop_density", "area_km2", "spatial_lag_turnout"]
feats = [f for f in feats if f in y21.columns]
corr = y21[feats].corr()

fig, ax = plt.subplots(figsize=(7.6, 6.4))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(feats))); ax.set_xticklabels(feats, rotation=45, ha="right")
ax.set_yticks(range(len(feats))); ax.set_yticklabels(feats)
for i in range(len(feats)):
    for j in range(len(feats)):
        v = corr.iloc[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7.5,
                color="white" if abs(v) > .55 else INK2)
ax.set_title("Correlations, 2021")
for s in ax.spines.values(): s.set_visible(False)
ax.grid(False)
fig.colorbar(im, ax=ax, shrink=.7, label="correlation")
save(fig, "12_correlations"); plt.show()

**What it shows.** Fill in the two or three strongest relationships from your own output.

**Why it matters.** It tells the model which features are worth keeping, and warns us about pairs
that say the same thing twice.

---
## 14. How much registration effort went in?

From the IEC Atlas: new registrations, re-registrations, and voting-district moves per
municipality, for the 2011 and 2016 registration weekends. This is published nowhere else.

It answers a question the dashboard cannot: not just *how full* a roll is, but *how much work*
went into filling it.

In [ ]:
act16 = pd.read_csv(INTERIM / "atlas_registration_activity_2016.csv")
a16 = master[master.election == 2016].merge(act16, on="muni_code", how="left")
a16["new_reg_rate"] = a16["NEWREG3"] / a16["registered"]

fig, ax = plt.subplots(figsize=(6.4, 5)) 
ax.scatter(a16["new_reg_rate"], a16["r"], s=30, color=BLUE,
           edgecolor=SURFACE, linewidth=.7, alpha=.85)
pct(ax); pct(ax, "x"); tidy(ax, xgrid=True)
ax.set_xlabel("new registrations as a share of the roll, 2016")
ax.set_ylabel("registration rate r, 2016")
ax.set_title("Did registration drives reach the places with the thinnest rolls?")
save(fig, "13_registration_effort"); plt.show()

print(a16[["new_reg_rate", "r"]].corr().round(3).to_string())

**What it shows.** Read the correlation printed above. A *negative* relationship would mean
effort went where the roll was thinnest — which is what you would want. A flat or positive one
would mean drives went where registration was already good.

**Why it matters.** It is a direct check on whether past registration effort was well targeted,
and it is exactly the kind of question our tool is meant to answer for 2026.

**Limitation.** The Atlas publishes this for 2011 and 2016 only. There is no equivalent for 2021
or 2026, so it **cannot** be a feature in the 2026 model — the column would be empty for the row
being predicted. We use it for this analysis only.

---
## 15. What we found

| # | Finding | Figure |
|---|---|---|
| 1 | The registration leak has been closing (63% → 79%) while the turnout leak has widened (60% → 48%). The two move in **opposite directions**. | `01` |
| 2 | Of every 100 eligible adults in 2021, about 65 registered and about 30 voted. The two losses are roughly equal, and only one of them is visible in published turnout. | `02` |
| 3 | Turnout fell in **every province** in 2021, pointing at a national cause rather than local ones. | `03` |
| 4 | Municipalities split into four clear groups by leak type. The same turnout figure can mean opposite problems. | `05` |
| 5 | Fifteen municipalities are below the national median on **both** leaks — the priority list for 4 November. | `06` |
| 6 | People aged 18–29 are ~30% of eligible adults but ~60% of everyone missing from the roll: about **6.5 million** young people. Every municipality is below parity. | `08` |
| 7 | Turnout **clusters geographically**, which justifies the spatial feature in the model. | `10` |

### What we are careful not to claim

- These are **associations**, never proven causes. Patterns across municipalities cannot
  establish how individuals behave.
- The youth finding is about **registration only**. The IEC does not publish turnout by age at
  municipal level, so we cannot say whether young people vote less once registered.
- The 2026 registration rate uses a 2022 population denominator and is overstated by roughly
  5%. This is a level bias, not a ranking bias — the order of municipalities is unaffected.
- `r` above 1 reflects a cumulative voters' roll against a point-in-time census, not an error in
  the calculation.

### Next

`08_model.ipynb` — predict the **change** in turnout for 2026, train on 2011→2016, test on
2016→2021, and replace the 2021 turnout used in §7 with a real forecast.

In [ ]:
print("figures written to", FIGURES)
for f in sorted(FIGURES.glob("*.png")):
    print(" ", f.name)